# MEIKURAL — choose a 30-second passage

Run the cells in order using **Python (MEIKURAL)**. Enter a song and trained checkpoint in the controls, then click **Load audio**. Move the start-time slider or enter exact seconds, listen with **Play selected clip**, and click **Predict selected clip**.

Only your selected passage is classified. Changing the selection clears previous results and does not run inference. Load audio estimates Sa once from the full song. Every slider selection reuses that fixed reference and the same Version 2 preprocessing used in training. No files or model weights are changed.

**Before use:** expand the Version 2 datasets, prepare features, and train `models/ver2/latest.pt` yourself. Version 1 checkpoints are rejected. Full-recording tonic estimation requires the whole supplied audio and can use substantial RAM; an incorrect estimate misaligns every window consistently.

**Shared test audio:** all versions use `data/test_ver1_handpicked`, including its `marwa` folder. Checkpoints, feature caches, and reports remain version-specific. After adding files, rerun the recording-list and batch cells (or Run All) before reading the confusion matrices. Saved outputs are not refreshed automatically.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src/ver2/config.json").is_file(), "Launch from the repository root or notebooks/"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ver2.config import load_config, resolve_path
from src.ver2.inference.interactive import clip_picker

config = load_config()


In [ ]:
# You can also change both paths directly in the interactive controls below.
TEST_DIR = ROOT / "data/test_ver1_handpicked"
AUDIO_PATH = TEST_DIR / 'sree/Venkatesh_Kumar.mp3'
CHECKPOINT_PATH = resolve_path(config["paths"]["run"]) / "latest.pt"

picker = clip_picker(AUDIO_PATH, CHECKPOINT_PATH)
display(picker)


In [ ]:
# You can also change both paths directly in the interactive controls below.
AUDIO_PATH = TEST_DIR / 'bhairavi/Bhimsen_Joshi.mp3'
CHECKPOINT_PATH = resolve_path(config["paths"]["run"]) / "latest.pt"

picker = clip_picker(AUDIO_PATH, CHECKPOINT_PATH)
display(picker)


## Reading the result

- The start time is relative to the original song. The selection normally spans **30 seconds**, as saved in your Version 2 checkpoint. A shorter song is padded with zeros on the right for the model; playback contains only the real audio.
- The last available start time ends the window at the song's end. For a different selection, move the slider and click the buttons again. Changing either file path requires clicking **Load audio** again.
- The ranking belongs to this single passage, without averaging predictions from other parts of the song. The full-recording Sa and device are shown with the result. A silence or tonic-extraction failure produces an explanation instead of a prediction.
- Listening helps you choose vocal passages, but this is **not vocal separation**. The CNN still sees accompaniment and harmonics. A confident-looking score or a successful tonic estimate can be wrong.
- These are comparative scores over seven supported raags, not calibrated probabilities or unknown-raag detection. Hand-selected passages used to guide model changes become development examples; keep independent recordings for final testing.

The song is decoded to mono at the checkpoint sample rate and held in RAM for quick seeking. Multi-hour files can require substantial memory. The model is loaded and Sa is estimated from the full recording once per **Load audio** click. Slider changes never re-estimate Sa. CUDA is selected when available; the displayed device confirms which is used.

If you see widget text rather than controls, run `python3 -m pip install -r requirements.txt` in the activated environment, then restart JupyterLab and select the correct kernel. A standalone static HTML export cannot run this Python model; the live notebook kernel is required.